# Workbench

**Try a circuit, run it, and compare it with the committed results, one cell per step.** The `Workbench` holds this session's settings and calls the package (`src/bellbench/`), which does all the work and writes every file.

- **Edit and run your copy in `workspace/`**, which `make workbench` makes. The notebook in `notebooks/` is generated; edits there are lost on the next render.
- **A circuit you build is a trial,** saved in `workspace/circuits/`, which is never committed. A run of a trial, kept, lands in `workspace/runs/`. When a trial is worth keeping, `promote` moves it into `config/circuit/`, ready to commit.
- **Each result shows itself:** F and acceptance, each basis's shots and correlator, the cost, the gates, what each check flagged, and a warning if its ideal run fails the self-test.

The cells: the import, the session's settings, then `fetch`, `build`, `show`, `run`, `compare`, `keep` and `promote`.

In [ ]:
# The one import. builders holds the transports and checks that build steps name.
from bellbench.workbench import Workbench, builders

In [ ]:
# The session's settings. Each defaults to the core benchmark's, so a baseline run repeats its seeds.
wb = Workbench(
    noise="boston-2026",
    # shots=4000,                  # per basis circuit, in each of the ideal and noisy simulations
    # scale=1.0,                   # multiplies every gate error, both readout errors and the decay rates
    # overrides=[{"qubits": [0, 1], "field": "gates.cz.error_probability", "value": 0.01}],
    # A circuit that resets a qubit, or reads one and acts on it again, needs the matching channel, swept over a
    # range rather than claimed at one value: {"reset_error": p, "mid_circuit_measurement_error": p}.
    # conditional={"reset_error": 0.01, "mid_circuit_measurement_error": 0.01},
)
BASELINE = "baseline-contemporary"  # the committed baseline the trial is compared with
wb

In [ ]:
# fetch: a committed run, shown without running it again. With analysis=..., its counts are re-read another way,
# labelled as a view.
base = wb.fetch("move-chain", BASELINE)
base

In [ ]:
# build: a trial circuit from builder steps, each a step or a (step, its arguments) pair. Building a name again
# replaces that trial. OpenQASM text or a Qiskit circuit works too, with its processor, and its analysis when it
# measures any bit:
# wb.build("my-pair", qasm_text, processor="ibm2016", analysis={"name": "postselect"})
wb.build("try-move-xx", "contemporary", builders.move_chain, (builders.bus_xx, {"on": [0, 4, 5, 6, 7]}))

In [ ]:
# show: the written circuit, the analysis that reads its shots, and the processor's graph with its ports.
wb.show("try-move-xx")

In [ ]:
# run: compile, simulate ideal and noisy, and score. Settings given here apply to this run only.
trial = wb.run("try-move-xx")
trial

In [ ]:
# compare: one row per result's noisy simulation, and whether its ideal run passed the self-test.
wb.compare(base, trial)

In [ ]:
# keep: write a result's run folder. A trial's lands in workspace/runs/, never committed.
folder = wb.keep(trial)
folder

In [ ]:
# promote: move a trial worth keeping into config/circuit/, ready to commit. A run kept from the trial can then be
# neither rerun nor fetched, since its trial file is gone; run the promoted circuit again to repeat it.
# wb.promote("try-move-xx")